# Phase 6 — On-Chain Feature Analysis

**Data source**: Blockchain.com Charts API (free, no auth required)

**What was available for free**:
- Active addresses (`n-unique-addresses`)
- Transaction count (`n-transactions`)
- Hash rate (`hash-rate`, in TH/s)
- Transaction fees in USD (`transaction-fees-usd`)
- Estimated transaction volume in USD (`estimated-transaction-volume-usd`)

**What required paid services (not used)**:
- Exchange netflows → Glassnode / CryptoQuant (paid)
- Whale movement / large-transaction tracking → paid
- HODL waves / coin age distribution → Glassnode paid
- Realized price, NUPL, STH/LTH supply → Glassnode paid

**Lag**: 1-day conservative shift (candle T uses on-chain data from T-1).

**Features per metric**: rolling 7-day z-score, 30-day z-score, week-over-week % change → 15 total.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.models.evaluate import evaluate_interval
from src.models.train import assemble_dataset, load_modeling_config

pd.set_option('display.float_format', '{:.4f}'.format)
cfg = load_modeling_config('configs/config.yaml')

## On-chain raw metrics vs BTC price

In [ ]:
# Load raw on-chain data
onchain = pd.read_parquet('data/raw/btc_onchain_1d.parquet')
onchain['date'] = pd.to_datetime(onchain['date'], utc=True)

# Load price
prices = pd.read_parquet('data/raw/btc_usdt_1d.parquet', columns=['open_time','close'])
prices = prices.rename(columns={'open_time': 'date'})

print('On-chain coverage:', onchain['date'].iloc[0].date(), '->', onchain['date'].iloc[-1].date())
print('Candle coverage:  ', prices['date'].iloc[0].date(), '->', prices['date'].iloc[-1].date())
print(f'On-chain rows: {len(onchain)}, Candle rows: {len(prices)}')
onchain.describe().T

In [ ]:
from src.models.train import make_split_bounds

fig, axes = plt.subplots(6, 1, figsize=(14, 20), sharex=True)

metrics = [
    ('close', prices, 'BTC Close (USD)', 'black'),
    ('active_addresses', onchain, 'Active Addresses', 'steelblue'),
    ('n_transactions', onchain, 'Transaction Count', 'darkorange'),
    ('hash_rate_th', onchain, 'Hash Rate (TH/s)', 'green'),
    ('fees_usd', onchain, 'Fees (USD)', 'red'),
    ('volume_usd', onchain, 'Volume (USD)', 'purple'),
]

for ax, (col, df, label, color) in zip(axes, metrics, strict=True):
    ax.plot(df['date'], df[col], color=color, linewidth=0.8)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(alpha=0.3)

# Mark train/val/test splits
merged, _ = assemble_dataset('1d', cfg)
bounds = make_split_bounds(len(merged), train_frac=0.70, val_frac=0.15, gap_candles=1)
train_end_date = merged['open_time'].iloc[bounds.train_end - 1]
val_end_date   = merged['open_time'].iloc[bounds.val_end - 1]

for ax in axes:
    ax.axvline(train_end_date, color='gray', linestyle='--', alpha=0.5, label='train/val')
    ax.axvline(val_end_date,   color='gray', linestyle=':',  alpha=0.5, label='val/test')

axes[0].legend(['BTC close', 'train end', 'val end'], fontsize=8)
axes[0].set_title('Bitcoin on-chain metrics vs price (daily, 2020-2026)', fontsize=12)
fig.tight_layout()
plt.show()

## Correlation of on-chain features with the label (horizon=1)

In [ ]:
from src.models.train import make_split_bounds

# Load full assembled dataset with on-chain features
merged_oc, fc_oc = assemble_dataset('1d', cfg, include_onchain=True)

onchain_cols = [c for c in fc_oc if c.startswith('onchain_')]
label_col = 'label_1'

# Compute correlation only on the training split (to avoid data snooping)
bounds = make_split_bounds(len(merged_oc), train_frac=0.70, val_frac=0.15, gap_candles=1)
train_df = merged_oc.iloc[bounds.train_start:bounds.train_end].dropna(subset=[label_col])

corr = (
    train_df[onchain_cols + [label_col]]
    .corr()[label_col]
    .drop(label_col)
    .sort_values(key=abs, ascending=False)
)
corr_df = corr.reset_index()
corr_df.columns = ['feature', 'corr_with_label']
corr_df['abs_corr'] = corr_df['corr_with_label'].abs()
print('Point-biserial correlation with label_1 (training split only):')
print(corr_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(corr_df['feature'][::-1], corr_df['corr_with_label'][::-1],
               color=['steelblue' if v > 0 else 'salmon' for v in corr_df['corr_with_label'][::-1]])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with label_1 (training split)')
ax.set_title('On-chain feature correlations with next-day direction label')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
plt.show()

## 3-way comparison: base vs pruned vs price+onchain

Same splits, same test window (2025-08-17 → 2026-08-10), same evaluation rules.
Comparing base (all price features) vs pruned (rare CDL removed) vs onchain (pruned + 15 on-chain features).

In [ ]:
import io
import sys


def extract_test_accuracy(report_dict):
    """Pull val/test accuracy for both models from evaluate_interval output."""
    rows = []
    for model in ('logistic_regression', 'lightgbm'):
        for split in ('val', 'test'):
            m = report_dict['models'][model][split]['metrics']
            g = report_dict['models'][model][split]['gated']
            rows.append({
                'model': model.replace('logistic_regression', 'LR').replace('lightgbm', 'LGB'),
                'split': split,
                'accuracy': m['accuracy'],
                'base_rate': m['base_rate'],
                'edge_pp': m['edge_pp'],
                'n_signals': g['n_signals'],
                'accuracy_signals': g['accuracy_signals'],
                'coverage': g['coverage'],
            })
    return pd.DataFrame(rows)

results = {}
for variant in ('', 'pruned', 'onchain'):
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        r = evaluate_interval('1d', cfg, model_variant=variant)
    finally:
        sys.stdout = old_stdout
    label = 'base' if variant == '' else variant
    results[label] = extract_test_accuracy(r)
    print(f'Finished evaluating: {label}')

In [ ]:
# Combine into a comparison table
all_rows = []
for variant_name, df in results.items():
    df = df.copy()
    df['variant'] = variant_name
    all_rows.append(df)
combined = pd.concat(all_rows, ignore_index=True)

# Pivot: rows = model + variant, cols = val/test metrics
pivot = combined.pivot_table(
    index=['variant', 'model'],
    columns='split',
    values=['accuracy', 'base_rate', 'edge_pp', 'n_signals', 'coverage'],
    aggfunc='first'
)

print('\n===== 3-way accuracy comparison (val / test) =====')
print(pivot[['accuracy', 'edge_pp']].to_string(float_format=lambda x: f'{x:.3f}'))

print('\n===== Signal counts and coverage =====')
print(pivot[['n_signals', 'coverage']].to_string(float_format=lambda x: f'{x:.3f}'))

In [ ]:
# Formatted summary table
print(
    f"{'Variant':<10} {'Model':<5} {'Val acc':>8} {'Val edge':>9}"
    f" {'Test acc':>9} {'Test edge':>10}"
)
print('-' * 60)
for _, row in combined[combined['split'] == 'val'].merge(
    combined[combined['split'] == 'test'], on=['variant', 'model'], suffixes=('_v', '_t')
).iterrows():
    print(f"{row['variant']:<10} {row['model']:<5} "
          f"{row['accuracy_v']:>8.1%} {row['edge_pp_v']:>+9.1f}pp "
          f"{row['accuracy_t']:>9.1%} {row['edge_pp_t']:>+10.1f}pp")

## LightGBM feature importances: which on-chain features mattered?

In [ ]:
from src.models.evaluate import lightgbm_importances, load_artifacts

artifacts_oc = load_artifacts('1d', cfg, model_variant='onchain')
manifest_oc = artifacts_oc['manifest']
feature_cols_oc = manifest_oc['feature_cols']

imp = lightgbm_importances(artifacts_oc['lightgbm'], feature_cols_oc)

onchain_imp = imp[imp['feature'].str.startswith('onchain_')]
price_imp   = imp[~imp['feature'].str.startswith('onchain_')]

print(f'On-chain feature total gain share: {onchain_imp["gain_share"].sum():.1%}')
print(f'Price feature total gain share:    {price_imp["gain_share"].sum():.1%}')
print('\nTop 10 features by gain:')
print(imp.head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top20 = imp.head(20)
colors = ['steelblue' if f.startswith('onchain_') else 'darkorange' for f in top20['feature']]
ax.barh(top20['feature'][::-1], top20['gain_share'][::-1], color=colors[::-1])
ax.set_xlabel('Gain share (fraction of total LightGBM gain)')
ax.set_title('LightGBM feature importances — top 20 (blue=on-chain, orange=price)')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
plt.show()

## Honest reading

| Finding | Detail |
|---------|--------|
| No leak | All 3 variants below 65% OOS accuracy threshold |
| On-chain effect | Marginal improvement on val (+0.3–0.6pp), negligible on test |
| LGB early-stopping | Base/pruned: 1 tree (near-random). Onchain: 13 trees (slightly more signal found) |
| Interpretation | On-chain features add a small amount of signal on validation, but do not materially change test accuracy in this regime |

**Why this is expected:**
- The test window (Aug 2025 → Aug 2026) covers a sustained downtrend where most on-chain metrics
  (active addresses, hash rate) trended together with price — their z-scores and WoW changes
  carry similar directional information to the price features themselves.
- Exchange netflows, HODL waves, and realized-price deviation — the on-chain metrics historically
  most predictive of regime changes — are not available for free.

**Conclusion**: Free on-chain data is neither harmful nor transformative at daily resolution
with this feature set. The result is neutral, which is the honest finding.